# Deep Learning Lizard Challenge - score boost

Deze notebook volgt de PowerPoint: EDA, eigen CNN, transfer learning, data augmentation, training plots, confusion matrix en een geldige Kaggle submission.

Score-focus: sterker EfficientNetV2-model, fine-tuning met lage learning rate, validatiegestuurde modelkeuze en TTA voor stabielere Kaggle-voorspellingen.

In [ ]:
from pathlib import Path
import random, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow", tf.__version__)

## 1. Data inlezen

In [ ]:
BASE_DIR = Path("lizard-prediction-thomas-more")
TRAIN_DIR = BASE_DIR / "train"
TEST_DIR = BASE_DIR / "test"

train_df = pd.read_csv(BASE_DIR / "train.csv")
test_df = pd.read_csv(BASE_DIR / "test.csv")
sample_sub = pd.read_csv(BASE_DIR / "sample_submission.csv")

CLASS_NAMES = sorted([p.name for p in TRAIN_DIR.iterdir() if p.is_dir()])
NUM_CLASSES = len(CLASS_NAMES)
print(len(train_df), "trainbeelden")
print(len(test_df), "testbeelden")
print(NUM_CLASSES, "klassen")
for i, name in enumerate(CLASS_NAMES):
    print(i, name)
train_df.head()

In [ ]:
def resolve_train_path(row):
    label = int(row["label"])
    class_dir = TRAIN_DIR / CLASS_NAMES[label]
    raw_id = str(row["id"])
    candidates = [class_dir / raw_id, class_dir / f"{raw_id}.jpg", class_dir / f"{raw_id}.jpeg", class_dir / f"{raw_id}.png"]
    for path in candidates:
        if path.exists():
            return str(path)
    stem = Path(raw_id).stem
    hits = list(class_dir.glob(stem + ".*"))
    if hits:
        return str(hits[0])
    raise FileNotFoundError(raw_id)

def resolve_test_path(image_id):
    for ext in [".jpg", ".jpeg", ".png"]:
        path = TEST_DIR / f"{int(image_id)}{ext}"
        if path.exists():
            return str(path)
    raise FileNotFoundError(image_id)

train_df["path"] = train_df.apply(resolve_train_path, axis=1)
test_df["path"] = test_df["id"].apply(resolve_test_path)
assert train_df["path"].map(lambda p: Path(p).exists()).all()
assert test_df["path"].map(lambda p: Path(p).exists()).all()
print("Alle image paths gevonden")

## 2. EDA

De klassen zijn licht uit balans. Daarom gebruiken we later `class_weight`.

In [ ]:
class_counts = train_df["label"].value_counts().sort_index()
eda = pd.DataFrame({"label": class_counts.index, "class_name": [CLASS_NAMES[i] for i in class_counts.index], "count": class_counts.values})
display(eda)

plt.figure(figsize=(10,4))
plt.bar(eda["class_name"], eda["count"])
plt.xticks(rotation=35, ha="right")
plt.title("Klasseverdeling")
plt.ylabel("Aantal afbeeldingen")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12,7))
for label in range(NUM_CLASSES):
    row = train_df[train_df["label"] == label].sample(1, random_state=SEED).iloc[0]
    img = keras.utils.load_img(row["path"], target_size=(180, 180))
    plt.subplot(2, 4, label + 1)
    plt.imshow(img)
    plt.title(f"{label}: {CLASS_NAMES[label]}", fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()

## 3. Stratified split en class weights

In [ ]:
train_split, val_split = train_test_split(train_df, test_size=0.20, random_state=SEED, stratify=train_df["label"])
print("train", len(train_split), "validatie", len(val_split))
print(val_split["label"].value_counts().sort_index())

weights = compute_class_weight(class_weight="balanced", classes=np.arange(NUM_CLASSES), y=train_split["label"].values)
CLASS_WEIGHTS = {i: float(w) for i, w in enumerate(weights)}
CLASS_WEIGHTS

## 4. TensorFlow datasets

EfficientNetV2 met `include_preprocessing=True` verwacht RGB-pixels in `[0,255]`.

In [ ]:
BATCH_SIZE = 16
IMG_SIZE = (300, 300)      # scoregericht; zet naar (260,260) als training te traag is
BACKBONE = "EfficientNetV2S"  # alternatief: "EfficientNetV2M" voor een zwaardere run
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label=None, img_size=IMG_SIZE):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image = tf.image.convert_image_dtype(image, tf.float32) * 255.0
    image = tf.image.resize(image, img_size)
    if label is None:
        return image
    return image, tf.cast(label, tf.int32)

def make_labeled_ds(df, img_size=IMG_SIZE, shuffle=False, batch_size=BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices((df["path"].values, df["label"].values))
    if shuffle:
        ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda p, y: load_image(p, y, img_size), num_parallel_calls=AUTOTUNE)
    return ds.batch(batch_size).prefetch(AUTOTUNE)

def make_unlabeled_ds(paths, img_size=IMG_SIZE, batch_size=BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices(np.array(paths, dtype=str))
    ds = ds.map(lambda p: load_image(p, None, img_size), num_parallel_calls=AUTOTUNE)
    return ds.batch(batch_size).prefetch(AUTOTUNE)

train_ds = make_labeled_ds(train_split, shuffle=True)
val_ds = make_labeled_ds(val_split)

## 5. Eigen CNN baseline

Deze baseline is vooral voor de minimumvereiste uit de opdracht. Het transfer-learning model zal normaal beter scoren.

In [ ]:
def build_own_cnn(input_shape, num_classes):
    return keras.Sequential([
        keras.Input(shape=input_shape),
        layers.Rescaling(1./255),
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.08),
        layers.RandomZoom(0.12),
        layers.Conv2D(32, 3, padding="same", activation="relu"), layers.BatchNormalization(), layers.MaxPooling2D(),
        layers.Conv2D(64, 3, padding="same", activation="relu"), layers.BatchNormalization(), layers.MaxPooling2D(),
        layers.Conv2D(128, 3, padding="same", activation="relu"), layers.BatchNormalization(), layers.MaxPooling2D(),
        layers.Conv2D(256, 3, padding="same", activation="relu"), layers.BatchNormalization(), layers.GlobalAveragePooling2D(),
        layers.Dropout(0.45),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.30),
        layers.Dense(num_classes, activation="softmax"),
    ], name="own_cnn_baseline")

cnn_model = build_own_cnn((*IMG_SIZE, 3), NUM_CLASSES)
cnn_model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cnn_model.summary()

In [ ]:
TRAIN_OWN_CNN = False
if TRAIN_OWN_CNN:
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=8, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3, min_lr=1e-6),
        keras.callbacks.ModelCheckpoint("scoreboost_own_cnn.keras", monitor="val_accuracy", save_best_only=True),
    ]
    history_cnn = cnn_model.fit(train_ds, validation_data=val_ds, epochs=35, class_weight=CLASS_WEIGHTS, callbacks=callbacks)
else:
    history_cnn = None
    print("Eigen CNN niet opnieuw getraind. Zet TRAIN_OWN_CNN=True als je dit wilt runnen.")

## 6. Transfer learning + fine-tuning

In [ ]:
def backbone_class(name):
    if name == "EfficientNetV2S":
        return keras.applications.EfficientNetV2S
    if name == "EfficientNetV2M":
        return keras.applications.EfficientNetV2M
    raise ValueError("Gebruik EfficientNetV2S of EfficientNetV2M")

def build_transfer_model(backbone_name=BACKBONE, img_size=IMG_SIZE):
    Backbone = backbone_class(backbone_name)
    base = Backbone(include_top=False, weights="imagenet", input_shape=(*img_size, 3), include_preprocessing=True)
    base.trainable = False
    augmentation = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.08),
        layers.RandomZoom(0.15),
        layers.RandomContrast(0.15),
        layers.RandomTranslation(0.05, 0.05),
    ], name="data_augmentation")
    inputs = keras.Input(shape=(*img_size, 3), name="image")
    x = augmentation(inputs)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.35)(x)
    x = layers.Dense(384, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.35)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="species")(x)
    return keras.Model(inputs, outputs, name=f"lizard_{backbone_name}_scoreboost"), base

model, base_model = build_transfer_model()
model.compile(optimizer=keras.optimizers.Adam(7e-4), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
TRAIN_TRANSFER = True
if TRAIN_TRANSFER:
    callbacks1 = [
        keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=8, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.35, patience=3, min_lr=1e-6),
        keras.callbacks.ModelCheckpoint("scoreboost_phase1.keras", monitor="val_accuracy", save_best_only=True),
    ]
    history_phase1 = model.fit(train_ds, validation_data=val_ds, epochs=25, class_weight=CLASS_WEIGHTS, callbacks=callbacks1)
else:
    history_phase1 = None

In [ ]:
if TRAIN_TRANSFER:
    model = keras.models.load_model("scoreboost_phase1.keras")
    backbone = next(layer for layer in model.layers if "efficientnet" in layer.name.lower())
    backbone.trainable = True
    fine_tune_last_n = 80 if BACKBONE == "EfficientNetV2S" else 110
    for layer in backbone.layers[:-fine_tune_last_n]:
        layer.trainable = False
    for layer in backbone.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

    lr_schedule = keras.optimizers.schedules.CosineDecay(8e-6, decay_steps=max(1, len(train_split)//BATCH_SIZE)*40, alpha=0.08)
    model.compile(optimizer=keras.optimizers.Adam(lr_schedule), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    callbacks2 = [
        keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True),
        keras.callbacks.ModelCheckpoint("scoreboost_finetuned.keras", monitor="val_accuracy", save_best_only=True),
    ]
    history_phase2 = model.fit(train_ds, validation_data=val_ds, epochs=40, class_weight=CLASS_WEIGHTS, callbacks=callbacks2)
else:
    history_phase2 = None

## 7. Training curves

In [ ]:
def plot_history(history, title):
    if history is None:
        print("Geen history voor", title)
        return
    hist = pd.DataFrame(history.history)
    fig, axes = plt.subplots(1, 2, figsize=(12,4))
    axes[0].plot(hist["accuracy"], label="train")
    axes[0].plot(hist["val_accuracy"], label="validatie")
    axes[0].set_title(title + " accuracy")
    axes[0].legend()
    axes[1].plot(hist["loss"], label="train")
    axes[1].plot(hist["val_loss"], label="validatie")
    axes[1].set_title(title + " loss")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

plot_history(history_cnn, "Eigen CNN")
plot_history(history_phase1, "Transfer fase 1")
plot_history(history_phase2, "Fine-tuning")

## 8. Modelkeuze en confusion matrix

We kiezen niet blind een ensemble. Als een ensemble op validatie slechter is dan het beste losse model, gebruiken we het beste losse model. Zo maken we de Kaggle-submission minder fragiel.

In [ ]:
def predict_model(model_path, paths, img_size):
    m = keras.models.load_model(model_path)
    probs = m.predict(make_unlabeled_ds(paths, img_size=img_size), verbose=0)
    tf.keras.backend.clear_session(); gc.collect()
    return probs

candidate_models = []
if Path("scoreboost_finetuned.keras").exists(): candidate_models.append(("scoreboost_finetuned.keras", IMG_SIZE))
if Path("scoreboost_phase1.keras").exists(): candidate_models.append(("scoreboost_phase1.keras", IMG_SIZE))
for item in [("best_efficientnetv2s_pwp.keras", (260,260)), ("phase1_best_efficientnetv2s_pwp.keras", (260,260)), ("best_model_fase2.keras", (224,224)), ("best_model_fase1.keras", (224,224))]:
    if Path(item[0]).exists(): candidate_models.append(item)

seen=set(); candidate_models=[x for x in candidate_models if not (x[0] in seen or seen.add(x[0]))]
print(candidate_models)

In [ ]:
val_paths = val_split["path"].tolist()
y_true = val_split["label"].to_numpy()
val_probs_list = []
rows = []
for path, size in candidate_models:
    probs = predict_model(path, val_paths, size)
    acc = accuracy_score(y_true, probs.argmax(axis=1))
    val_probs_list.append(probs)
    rows.append({"model": path, "img_size": size, "val_accuracy": acc})

scores = pd.DataFrame(rows).sort_values("val_accuracy", ascending=False)
display(scores)
best_model_path = scores.iloc[0]["model"]
best_img_size = tuple(scores.iloc[0]["img_size"])
best_index = candidate_models.index((best_model_path, best_img_size))
best_val_probs = val_probs_list[best_index]
best_val_acc = scores.iloc[0]["val_accuracy"]

accs = np.array([r["val_accuracy"] for r in rows])
weights = (accs ** 2) / np.sum(accs ** 2)
ensemble_val_probs = sum(w*p for w, p in zip(weights, val_probs_list))
ensemble_val_acc = accuracy_score(y_true, ensemble_val_probs.argmax(axis=1))
USE_ENSEMBLE = ensemble_val_acc > best_val_acc
print("Best single:", best_model_path, round(best_val_acc,4))
print("Ensemble:", round(ensemble_val_acc,4), "use?", USE_ENSEMBLE)

final_val_probs = ensemble_val_probs if USE_ENSEMBLE else best_val_probs
y_pred = final_val_probs.argmax(axis=1)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8,8))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.tight_layout(); plt.show()

## 9. Kaggle submission met TTA

In [ ]:
def predict_with_tta(model_path, paths, img_size, num_tta=5):
    m = keras.models.load_model(model_path)
    base_ds = make_unlabeled_ds(paths, img_size=img_size)
    all_probs = []
    for tta_i in range(num_tta):
        parts = []
        for batch in base_ds:
            x = batch
            if tta_i == 1:
                x = tf.image.flip_left_right(batch)
            elif tta_i == 2:
                x = tf.image.adjust_contrast(batch, 1.08)
            elif tta_i == 3:
                x = tf.image.adjust_brightness(batch, 8.0)
            elif tta_i == 4:
                margin_h = max(1, int(img_size[0] * 0.04))
                margin_w = max(1, int(img_size[1] * 0.04))
                cropped = tf.image.crop_to_bounding_box(batch, margin_h, margin_w, img_size[0] - 2*margin_h, img_size[1] - 2*margin_w)
                x = tf.image.resize(cropped, img_size)
            parts.append(m.predict(x, verbose=0))
        all_probs.append(np.concatenate(parts, axis=0))
    tf.keras.backend.clear_session(); gc.collect()
    return np.mean(all_probs, axis=0)

test_paths = test_df["path"].tolist()
NUM_TTA = 5

if USE_ENSEMBLE:
    probs = 0
    for (path, size), weight in zip(candidate_models, weights):
        print("predict", path, "weight", round(float(weight),3))
        probs = probs + weight * predict_with_tta(path, test_paths, size, NUM_TTA)
    output_name = "submission_scoreboost_ensemble_tta.csv"
else:
    print("predict beste model", best_model_path)
    probs = predict_with_tta(best_model_path, test_paths, best_img_size, NUM_TTA)
    output_name = "submission_scoreboost_bestmodel_tta.csv"

submission = pd.DataFrame({"id": sample_sub["id"], "label": probs.argmax(axis=1).astype(int)})
assert list(submission.columns) == list(sample_sub.columns)
assert len(submission) == len(sample_sub)
assert submission["label"].between(0, NUM_CLASSES-1).all()
submission.to_csv(output_name, index=False)
submission.to_csv("submission.csv", index=False)
print("saved", output_name, "and submission.csv")
display(submission.head(10))
print(submission["label"].value_counts().sort_index())

## 10. Conclusie voor de defense

- De eigen CNN toont convolution, pooling/GAP en classificatie uit de les.
- Transfer learning gebruikt een pretrained EfficientNetV2-body en een eigen classifier head.
- Fine-tuning unfreezet enkel de laatste lagen met een lage learning rate, zodat de ImageNet-kennis niet kapot getraind wordt.
- Data augmentation helpt tegen overfitting.
- De confusion matrix toont welke hagedissoorten nog verward worden.
- TTA maakt de testvoorspellingen stabieler voor Kaggle.

## GenAI-vermelding

AI werd gebruikt om de PowerPointvereisten te controleren en om een verbeterde TensorFlow/Keras-notebook te structureren. De prompts vroegen om de bestaande aanpak te verbeteren zonder de opdrachtregels te breken. De beperking is dat een hogere validatiescore geen garantie geeft op een hogere Kaggle-score; de uiteindelijke CSV moet dus op Kaggle getest worden. Ik heb de code nagekeken en kan de gebruikte technieken uitleggen.